<a href="https://colab.research.google.com/github/mehanshbarthwal-lab/search-ranking-ml/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mehanshbarthwal-lab/search-ranking-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

- Lane: freestyle, building on the Refresh / Content Opportunity Scoring lane.
- Title for the project: "Answered Away: Detecting and Costing Click-Suppression in Declining Content"


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
**Lane: freestyle**, building on the Refresh / Content Opportunity Scoring lane.

**Project title: "Answered Away: Detecting and Costing Click-Suppression in Declining Content"**

I'm going freestyle rather than picking one of the four predefined lanes because the question I
want to ask is narrower than any single lane covers. Instead of treating "declining" as one
problem, I want to split it into two behaviorally distinct patterns: pages losing traffic because
fewer people see them at all (normal decay), versus pages that keep showing up in search just as
often but stop getting clicked (a pattern consistent with something answering the query before
the click happens, which I'm calling "answered_away"). This builds directly on the Refresh /
Content Opportunity lane's mechanics, but adds a diagnostic split and an economics layer neither
the predefined lanes nor the freestyle "AI Referral Opportunity" direction attempt, since that
direction is explicitly EDA-only due to how sparse ai_sessions data is (only ~6% of pages have
any AI-referred sessions at all in the starter CSV). My approach avoids that sparsity problem
entirely by using impressions and clicks, which every page has, as the behavioral proxy instead.

SyntaxError: unterminated string literal (detected at line 7) (720820201.py, line 7)

## 2. The question: decision, action, cost of a wrong call

- **Core question:** when a page is losing clicks, is it losing them because fewer people see it at all (normal decay — impressions AND clicks both fall), or because roughly the same number of people see it but stop clicking (a pattern consistent with something answering the query before the click happens — impressions flat/up, clicks fall meaningfully). Call this second pattern "answered_away" — explicitly a behavioral pattern label, not a confirmed AI-Overview label, since there's no column anywhere confirming what caused it.
- **Decision this improves:** which declining pages should a content editor fix, and with which kind of fix — a routine refresh (normal_decay) vs a structural fix like FAQ blocks or a clearer direct-answer section (answered_away).
- **Action:** an editor pulls the ranked, labeled list and picks the right fix type per page.
- **Cost of a wrong call:** applying a structural fix to a page that just needed routine freshening wastes editor effort; missing a real answered_away page means it keeps losing clicks even after a refresh that was never going to fix the actual problem.
- **Why this earns ML over a hand-written rule:** a rule can flag the pattern after it's already happened; predicting which pages are at elevated risk of the answered_away pattern before it fully shows up, using several tangled prior-period signals together (intent, position, competition, content type, length), is where ML earns its place.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

- **Build the label from real data:** impr_change_pct and click_change_pct computed from impressions_last_30d vs impressions_prev_30d and clicks_last_30d vs clicks_prev_30d. Filter to pages with impressions_prev_30d >= 50 and clicks_prev_30d >= 3 (enough prior volume to trust the percent change). answered_away = impr_change_pct >= -5 AND click_change_pct <= -15. normal_decay = both impr_change_pct <= -15 AND click_change_pct <= -15. Everything else = stable_other.
- **Economics layer:** organic and paid search are substitute goods for the same click; cpc is the shadow price of that substitution. Compute ad-equivalent value at risk for the answered_away group specifically: (clicks_prev_30d - clicks_last_30d) * cpc, summed. State clearly this is a proxy for what it would cost to replace that traffic via ads, not a literal revenue loss, since we have no conversion data.


In [ ]:
import pandas as pd

# Load the starter CSV fresh
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Compute percentage changes
df['impr_change_pct'] = (df['impressions_last_30d'] - df['impressions_prev_30d']) / df['impressions_prev_30d'] * 100
df['click_change_pct'] = (df['clicks_last_30d'] - df['clicks_prev_30d']) / df['clicks_prev_30d'] * 100

# Filter to pages with enough prior volume to trust the percent change
volume_mask = (df['impressions_prev_30d'] >= 50) & (df['clicks_prev_30d'] >= 3)
df_filtered = df[volume_mask].copy()

# Define the pattern groups
def assign_pattern(row):
    if row['impr_change_pct'] >= -5 and row['click_change_pct'] <= -15:
        return 'answered_away'
    elif row['impr_change_pct'] <= -15 and row['click_change_pct'] <= -15:
        return 'normal_decay'
    else:
        return 'stable_other'

df_filtered['pattern_group'] = df_filtered.apply(assign_pattern, axis=1)

# Count total pages in each pattern group
print("Total pages in each pattern group:")
print(df_filtered['pattern_group'].value_counts())
print("\n")

# Compute ad-equivalent value at risk for the answered_away group
# (clicks_prev_30d - clicks_last_30d) * cpc, summed
answered_away_df = df_filtered[df_filtered['pattern_group'] == 'answered_away'].copy()
answered_away_df['ad_equivalent_value_at_risk'] = (answered_away_df['clicks_prev_30d'] - answered_away_df['clicks_last_30d']) * answered_away_df['cpc']
total_value_at_risk = answered_away_df['ad_equivalent_value_at_risk'].sum()

print(f"Ad-equivalent value at risk for 'answered_away' group: ${total_value_at_risk:,.2f}")
print("(Note: This is a proxy for what it would cost to replace that traffic via ads, not a literal revenue loss)")


## 4. Careful words: what I can and can't claim

- **Can claim:** a pattern consistent with click suppression, an association between page profile and pattern type, a proxy dollar value.
- **Can never claim:** that AI Overviews specifically caused anything, real revenue loss, causal proof — no controlled experiment is possible on historical data.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
